# Chapter 6 assignment: terminal value problems, conditioning and adjoints

Companion notebook for Chapter 6 of *Physics-Informed Neural Networks for Ordinary, Integral and
Integro-Differential Equations*. Everything runs on a CPU in a few minutes.

**Learning goals**

1. Integrate a terminal value problem (TVP) backward with a classical solver and through the time
   reversal $s = T - t$.
2. Quantify the backward amplification of errors with the matrix exponential and with perturbed
   terminal data.
3. Build a PINN with a *terminal trial function* that satisfies the terminal data exactly.
4. Compute a parameter gradient with the adjoint equation and check it against the exact value.

Cells marked `# TODO` are for you. The *check* cells print `OK` when your answer is correct and a
hint otherwise; they can be run before you finish (they then report `TODO`).

## 0. Setup

Run this notebook from the repository root (or add it to `sys.path`) so that `pinnode` can be imported.

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd()
if not (root / "pinnode").exists() and (root.parent / "pinnode").exists():
    root = root.parent
sys.path.insert(0, str(root))

import numpy as np
import torch
from scipy.integrate import solve_ivp
from scipy.linalg import expm
from pinnode import MLP, ConstrainedModel, derivatives, to_tensor, train, rel_l2, uniform, configure_torch, set_seed

configure_torch("float64", threads=1)
set_seed(1234)
print("torch", torch.__version__)

## 1. Theory recap

* A TVP $y' = f(t, y)$, $y(T) = y_T$ becomes an IVP for $Y(s) = y(T - s)$:
  $Y' = -f(T - s, Y)$, $Y(0) = y_T$. For a second-order equation $y'' = F(t, y, y')$ with
  $y(T) = c_0$, $y'(T) = c_1$: $Y'' = F(T - s, Y, -Y')$, $Y(0) = c_0$, $Y'(0) = -c_1$.
* For a linear system $z' = Az$ the terminal-to-initial map is $z(0) = e^{-AT} z(T)$; its norm is the
  amplification factor of errors in the terminal data. For dissipative $A$ it can be huge
  (Proposition "Backward stability estimate" in the chapter).
* Terminal trial function for an $n$-th order TVP:
  $y_\theta(t) = \sum_{k<n} c_k (t-T)^k / k! + ((T - t)/T)^n N_\theta(t)$.
* Adjoint: for $y' = f(y, p)$, $y(0) = y_0$ and $J = \int_0^T g(y)\,dt$, the adjoint
  $\lambda' = -f_y^\top \lambda - g_y^\top$, $\lambda(T) = 0$ gives
  $dJ/dp = \int_0^T \lambda^\top f_p\,dt$.

## 2. Task 1: backward integration and time reversal

Damped oscillator TVP: $y'' + 0.5\,y' + 4\,y = 0$ on $[0, 3]$, $y(3) = 0.2$, $y'(3) = -0.1$.

(a) Integrate the first-order system **backward** from $t = 3$ to $t = 0$ with `solve_ivp`
(`method="DOP853"`, `rtol=atol=1e-12`) and store $y$ on the grid `t` (increasing order) in `y_back`.

(b) Integrate the **reversed IVP** in $s = 3 - t$ forward and store $y(t)$ on the same grid in `y_rev`.

In [ ]:
T1, c0, c1 = 3.0, 0.2, -0.1
t = np.linspace(0.0, T1, 301)

def rhs(t, z):                     # z = (y, y')
    return [z[1], -0.5 * z[1] - 4.0 * z[0]]

y_back = None   # TODO (a): backward integration; remember that t_eval must decrease
y_rev = None    # TODO (b): reversed IVP Y'' = -0.5*(-Y') - 4 Y, Y(0) = c0, Y'(0) = -c1

In [ ]:
# check your answer: compare with the matrix exponential z(t) = expm(A (t - T)) z(T)
A1 = np.array([[0.0, 1.0], [-4.0, -0.5]])
y_exact = np.array([(expm(A1 * (ti - T1)) @ np.array([c0, c1]))[0] for ti in t])
for name, y in [("y_back", y_back), ("y_rev", y_rev)]:
    if y is None:
        print(f"{name}: TODO")
    else:
        err = np.max(np.abs(np.asarray(y) - y_exact))
        print(f"{name}: max error {err:.1e}", "OK" if err < 1e-9 else "-> check the direction of t_eval and the signs")

## 3. Task 2: how much does the backward direction amplify errors?

For $y'' + (1 + \kappa) y' + \kappa y = 0$ on $[0, 1]$ (modes $e^{-t}$ and $e^{-\kappa t}$) write
$z' = A z$ with $A = \begin{pmatrix} 0 & 1 \\ -\kappa & -(1+\kappa) \end{pmatrix}$.

(a) For $\kappa = 2, 5, 10, 20$ compute the backward factor $\|e^{-A}\|_2$ and the forward factor
$\|e^{A}\|_2$ (`np.linalg.norm(M, 2)`), and store them in the lists `amp_back`, `amp_fwd`.

(b) For $\kappa = 10$, perturb the terminal data $z(1)$ by $\delta v$, $\delta = 10^{-6}$, where $v$ is the
first right singular vector of $e^{-A}$, integrate backward with DOP853 and store the measured
amplification $\|z_\delta(0) - z(0)\| / \delta$ in `amp_measured`.

In [ ]:
kappas = [2.0, 5.0, 10.0, 20.0]
amp_back = None      # TODO (a)
amp_fwd = None       # TODO (a)
amp_measured = None  # TODO (b)

In [ ]:
# check your answer
ref_back = [np.linalg.norm(expm(-np.array([[0, 1], [-k, -(1 + k)]])), 2) for k in kappas]
if amp_back is None or amp_fwd is None:
    print("(a) TODO")
else:
    ok = np.allclose(amp_back, ref_back, rtol=1e-8) and max(amp_fwd) < 1.0
    print("(a)", "OK" if ok else "-> use expm(-A) for the backward and expm(A) for the forward map")
if amp_measured is None:
    print("(b) TODO")
else:
    print("(b)", "OK" if abs(amp_measured / ref_back[2] - 1) < 1e-4 else "-> perturb along the first right singular vector")

## 4. Task 3: a PINN with a terminal trial function

Solve the TVP of Task 1 with a PINN. Use `MLP(1, 1, width=32, depth=3, domain=(0, 3))` inside a
`ConstrainedModel` whose transform implements the terminal trial function
$y_\theta(t) = c_0 + c_1 (t - 3) + ((3 - t)/3)^2 N_\theta(t)$, collocation points `uniform(100, 0, 3)`,
and `train(..., adam_iters=3000, lr=1e-3, lr_final=1e-5, lbfgs_iters=300)`.
Store the prediction on the grid `t` in `y_pinn`.

In [ ]:
y_pinn = None
# TODO: build the model, the residual loss  r = y'' + 0.5 y' + 4 y  and train it
# model = ConstrainedModel(MLP(...), lambda tt, n: ...)
# t_col = to_tensor(uniform(100, 0.0, T1), requires_grad=True)
# def loss_fn():
#     y, dy, ddy = derivatives(model(t_col), t_col, 2)
#     ...
#     return L, {"residual": L}
# hist = train(model.parameters(), loss_fn, adam_iters=3000, lr=1e-3, lr_final=1e-5, lbfgs_iters=300)
# with torch.no_grad():
#     y_pinn = model(to_tensor(t)).numpy().ravel()

In [ ]:
# check your answer
if y_pinn is None:
    print("TODO")
else:
    e = rel_l2(y_pinn, y_exact)
    print(f"relative L2 error {e:.2e}", "OK" if e < 1e-3 else "-> train longer or check the trial function")
    print("terminal data:", y_pinn[-1], "(should be", c0, ")")

**Question.** The oscillator is only weakly damped. Compute $\|e^{-3A_1}\|_2$ for the matrix `A1` of
Task 1. Would you expect the PINN error to grow strongly towards $t = 0$? Compare with the
two-mode problem of Task 2 for $\kappa = 10$.

## 5. Task 4: adjoint gradient

For $y' = -p\,y$, $y(0) = 1$, and $J(p) = \tfrac12 \int_0^T y^2\,dt$ with $T = 2$:

(a) Show that the adjoint TVP is $\lambda' = p\lambda - y$, $\lambda(T) = 0$, and that
$dJ/dp = -\int_0^T \lambda\,y\,dt$.

(b) Implement the classical adjoint method: use the exact state $y = e^{-pt}$, integrate $\lambda$
backward with `solve_ivp` (DOP853, `dense_output=True`), and evaluate the integral with a
Gauss-Legendre rule (`np.polynomial.legendre.leggauss`). Store the result for $p = 0.7$ in `dJdp_adjoint`.

In [ ]:
p, T2 = 0.7, 2.0
dJdp_adjoint = None   # TODO (b)

In [ ]:
# check your answer: J(p) = (1 - exp(-2 p T)) / (4 p)
def J_exact(q):
    return (1.0 - np.exp(-2.0 * q * T2)) / (4.0 * q)
h = 1e-6
dJdp_fd = (J_exact(p + h) - J_exact(p - h)) / (2 * h)
if dJdp_adjoint is None:
    print("TODO")
else:
    rel = abs(dJdp_adjoint - dJdp_fd) / abs(dJdp_fd)
    print(f"adjoint {dJdp_adjoint:.10f}  finite differences {dJdp_fd:.10f}  rel. diff {rel:.1e}",
          "OK" if rel < 1e-7 else "-> check the sign of the adjoint equation and of dJ/dp")

## 6. Going further

* Solve the coupled state-adjoint system of Task 4 with **one** PINN with two outputs, using the
  trial functions $y_\theta = 1 + (t/T) N_1$ and $\lambda_\theta = ((T - t)/T) N_2$ (compare with
  `chapters/ch06_tvp/adjoint_sensitivity.py`).
* Repeat Task 3 for the two-mode problem of Task 2 with $\kappa = 10$ posed as a TVP and as an IVP.
  Plot both errors on a logarithmic scale and relate the slope of the TVP error to $e^{\kappa(T - t)}$
  (compare with `chapters/ch06_tvp/tvp_conditioning.py`).